# External Feature Importance Summary

Loads final models per department (cases/trucks), lists stack components, and surfaces top features from models that expose importances/coefs. Also highlights external features among the top signals.


In [1]:

import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.ensemble import StackingRegressor

# Load splits
splits_dir = Path('data/processed/timeseries_splits')
train = pd.read_csv(splits_dir / 'train_timeseries.csv', parse_dates=['dt'], low_memory=False)

# External feature patterns (subset of columns we consider external)
external_tokens = [
    'CPI_', 'Consumer_Sentiment', 'Initial_Jobless_Claims', 'Fed_Funds_Rate', 'Crude_Oil_Price',
    'state_unemployment_rate', 'credit_spend_index_all', 'gas_price_usd_per_gallon',
    'trends_', 'trend_', 'sales_tax', 'holiday', 'bts_', 'sports_event', 'is_school',
    'temp_', 'precip', 'cdd_', 'hdd_', 'cooling_degree_days', 'heating_degree_days'
]


def get_feature_columns(df, target):
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if target == 'cases':
        return [c for c in num_cols if c != 'cases']
    if target == 'trucks':
        return [c for c in num_cols if c != 'trucks' and not c.startswith('cases')]
    raise ValueError('target must be cases or trucks')


def top_features(model, feature_names, k=15):
    if hasattr(model, 'feature_importances_'):
        vals = model.feature_importances_
    elif hasattr(model, 'coef_'):
        vals = np.array(model.coef_).ravel()
    else:
        return None
    s = pd.Series(vals, index=feature_names)
    return s.abs().sort_values(ascending=False).head(k)


def is_external(feature):
    return any(tok in feature for tok in external_tokens)


rows = []
for dept in [6,9,41,67,90]:
    dept_dir = Path(f'models/dept{dept}')
    for target_key, target in [('case','cases'), ('truck','trucks')]:
        model_path = dept_dir / ('final_case_model.pkl' if target=='cases' else 'final_truck_model.pkl')
        if not model_path.exists():
            continue
        model = joblib.load(model_path)
        train_df = train[train['dept_id']==dept].copy()
        feature_cols = get_feature_columns(train_df, target)

        stack_components = None
        if isinstance(model, StackingRegressor):
            stack_components = [name for name, _ in model.estimators]
            # Try to pull top features from base estimators that support it
            base_feats = []
            for name, est in model.estimators:
                tf = top_features(est, feature_cols)
                if tf is not None:
                    base_feats.append(tf.rename(name))
            base_imp = pd.concat(base_feats, axis=1) if base_feats else None
        else:
            stack_components = []
            base_imp = None

        imp = top_features(model, feature_cols)
        ext_imp = None
        if imp is not None:
            ext_imp = imp[imp.index.map(is_external)]

        rows.append({
            'dept': dept,
            'target': target,
            'model_type': type(model).__name__,
            'is_stacker': isinstance(model, StackingRegressor),
            'stack_components': stack_components,
            'top_features': imp if imp is not None else 'n/a',
            'top_external_features': ext_imp if ext_imp is not None else 'n/a',
            'base_importances': base_imp if stack_components and base_imp is not None else 'n/a'
        })

summary = pd.DataFrame(rows)
summary[['dept','target','model_type','is_stacker','stack_components']]


,dept,target,model_type,is_stacker,stack_components
0,6,cases,CatBoostRegressor,False,[]
1,6,trucks,StackingRegressor,True,"[hgb, rf, lgbm]"
2,9,cases,StackingRegressor,True,"[cat, hgb, lgbm]"
3,9,trucks,StackingRegressor,True,"[lgbm, hgb, rf]"
4,41,cases,StackingRegressor,True,"[rf, cat, et]"
5,41,trucks,StackingRegressor,True,"[hgb, cat, rf]"
6,67,cases,CatBoostRegressor,False,[]
7,67,trucks,StackingRegressor,True,"[rf, lgbm, cat]"
8,90,cases,StackingRegressor,True,"[et, hgb, rf]"
9,90,trucks,StackingRegressor,True,"[hgb, rf, lgbm]"


## Top features by dept/target

In [3]:

for _, row in summary.iterrows():
    print(f"Dept {row.dept} | {row.target} | {row.model_type} | stackers: {row.stack_components}")
    if isinstance(row.top_features, pd.Series):
        display(row.top_features)
    else:
        print('No importances available')
    if isinstance(row.top_external_features, pd.Series) and not row.top_external_features.empty:
        print('Top external among them:')
        display(row.top_external_features)


Dept 6 | cases | CatBoostRegressor | stackers: []


trucks                          17.431522
cases_lag_7                     16.618515
cases_lag_14                    14.477598
cases_lag_1                      6.434414
cases_ma_7                       4.487025
cases_ma_14                      4.385068
trucks_ma_7                      4.053763
Crude_Oil_Price                  3.648573
Initial_Jobless_Claims           3.542775
trucks_lag_7                     2.573692
trucks_lag_1                     2.144220
cases_std_7                      1.371411
trends_sporting_goods_scaled     1.250023
Consumer_Sentiment               1.207752
trends_cameras_scaled            1.197793
dtype: float64

Top external among them:


Crude_Oil_Price                 3.648573
Initial_Jobless_Claims          3.542775
trends_sporting_goods_scaled    1.250023
Consumer_Sentiment              1.207752
trends_cameras_scaled           1.197793
dtype: float64

Dept 6 | trucks | StackingRegressor | stackers: ['hgb', 'rf', 'lgbm']
No importances available
Dept 9 | cases | StackingRegressor | stackers: ['cat', 'hgb', 'lgbm']
No importances available
Dept 9 | trucks | StackingRegressor | stackers: ['lgbm', 'hgb', 'rf']
No importances available
Dept 41 | cases | StackingRegressor | stackers: ['rf', 'cat', 'et']
No importances available
Dept 41 | trucks | StackingRegressor | stackers: ['hgb', 'cat', 'rf']
No importances available
Dept 67 | cases | CatBoostRegressor | stackers: []


cases_lag_7                     15.970563
cases_lag_14                    15.906551
trucks                          14.871623
cases_lag_1                      6.145851
cases_ma_14                      5.705202
cases_ma_7                       5.010358
trucks_ma_7                      4.082512
Crude_Oil_Price                  3.415642
trucks_lag_7                     3.404156
trucks_lag_1                     2.662283
Initial_Jobless_Claims           2.623603
trends_cameras_scaled            1.808969
trends_team_sports_scaled        1.601264
trends_sporting_goods_scaled     1.221419
Consumer_Sentiment               1.190959
dtype: float64

Top external among them:


Crude_Oil_Price                 3.415642
Initial_Jobless_Claims          2.623603
trends_cameras_scaled           1.808969
trends_team_sports_scaled       1.601264
trends_sporting_goods_scaled    1.221419
Consumer_Sentiment              1.190959
dtype: float64

Dept 67 | trucks | StackingRegressor | stackers: ['rf', 'lgbm', 'cat']
No importances available
Dept 90 | cases | StackingRegressor | stackers: ['et', 'hgb', 'rf']
No importances available
Dept 90 | trucks | StackingRegressor | stackers: ['hgb', 'rf', 'lgbm']
No importances available


## Base estimator importances (if stacker)

In [4]:

for _, row in summary.iterrows():
    if row.is_stacker and isinstance(row.base_importances, pd.DataFrame):
        print(f"Dept {row.dept} | {row.target} stack base models")
        display(row.base_importances)


Dept 6 | trucks stack base models


,rf,lgbm
trucks_lag_7,0.705327,NaN
trucks_lag_1,0.052935,NaN
trucks_ma_7,0.047842,2811.0
Crude_Oil_Price,0.020693,5266.0
Initial_Jobless_Claims,0.016027,1373.0
temp_min_f,0.013260,2965.0
temp_max_f,0.013142,2892.0
store_id,0.011177,2436.0
trends_team_sports_scaled,0.010786,NaN
gas_price_usd_per_gallon,0.010588,1851.0


Dept 9 | cases stack base models


,cat,lgbm
cases_lag_7,17.601734,2464.0
trucks,16.310495,NaN
cases_lag_14,14.799384,2587.0
cases_ma_14,6.471445,2354.0
cases_lag_1,5.574143,2130.0
Crude_Oil_Price,5.496147,3539.0
cases_ma_7,4.371953,2280.0
trucks_lag_7,3.973949,NaN
trucks_ma_7,3.924728,NaN
Initial_Jobless_Claims,2.777702,1284.0


Dept 9 | trucks stack base models


...

,rf,lgbm,cat
trucks_lag_7,0.705102,NaN,54.251376
trucks_lag_1,0.054186,NaN,8.591774
trucks_ma_7,0.046624,2344.0,9.006858
Crude_Oil_Price,0.021423,4214.0,4.841690
Initial_Jobless_Claims,0.016371,962.0,2.820645
temp_max_f,0.012964,2090.0,NaN
temp_min_f,0.012821,2057.0,NaN
store_id,0.011121,1830.0,1.222227
gas_price_usd_per_gallon,0.010583,1400.0,0.951453
trends_team_sports_scaled,0.010331,NaN,NaN


Dept 90 | cases stack base models


,et,rf
trucks,0.228885,0.474204
trucks_lag_7,0.109966,0.011956
cases_ma_14,0.107467,0.045165
cases_lag_7,0.087366,0.200701
cases_ma_7,0.086177,0.031788
trucks_ma_7,0.079550,0.018070
cases_lag_14,0.061405,0.071030
trucks_lag_1,0.032033,NaN
cases_lag_1,0.029149,0.023578
trends_cameras_scaled,0.009256,NaN


Dept 90 | trucks stack base models


,rf,lgbm
trucks_lag_7,0.705054,NaN
trucks_lag_1,0.054492,NaN
trucks_ma_7,0.046779,3000.0
Crude_Oil_Price,0.021416,5306.0
Initial_Jobless_Claims,0.016944,1479.0
temp_max_f,0.012501,2966.0
temp_min_f,0.012408,2904.0
store_id,0.011119,2615.0
gas_price_usd_per_gallon,0.010550,1837.0
trends_team_sports_scaled,0.010361,NaN


## Notes
- External tokens matched: CPI, sentiment, jobless claims, Fed funds, crude oil, state unemployment, credit spend, gas price, trends, sales_tax/holiday/event flags, school/bts, weather (temp/precip/CDD/HDD).
- Models without importances/coefs will show `n/a`.
